In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2
from pathlib import Path

video_path = Path("/content/drive/MyDrive/dataset-wajah-mahasiswa/IMG_2808.MOV")
output_root = Path("/content/drive/MyDrive/dataset-gabungan")

output_root.mkdir(parents=True, exist_ok=True)

min_frames = 50

cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    print("Video tidak terbaca. Cek path file video.")
else:
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    interval = max(total_frames // min_frames, 1)

    print("FPS:", fps)
    print("Total frames:", total_frames)
    print("Interval:", interval)

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_count % interval == 0:
            filename = output_root / f"IMG_2808_frame_{saved_count:04d}.jpg"
            cv2.imwrite(str(filename), frame)
            saved_count += 1

        frame_count += 1

    cap.release()

    print("Total saved:", saved_count)
    print("Selesai extract frame.")

FPS: 59.93961579721525
Total frames: 9033
Interval: 180
Total saved: 51
Selesai extract frame.


In [ ]:
import os

root = "/content/drive/MyDrive/dataset-emosion"

for folder in os.listdir(root):
    print(folder)

.DS_Store
bingung
fokus
senang
bosan


In [ ]:
import cv2
from pathlib import Path

# ======================
# INPUT VIDEO
# ======================

input_root = Path(
    "/content/drive/MyDrive/dataset-emosion"
)

# ======================
# OUTPUT FRAME
# ======================

output_root = Path(
    "/content/drive/MyDrive/dataset-frame-raw"
)

output_root.mkdir(
    parents=True,
    exist_ok=True
)

# ======================
# EKSTRAK FRAME
# ======================

for emotion_folder in input_root.iterdir():

    if not emotion_folder.is_dir():
        continue

    emotion_name = emotion_folder.name

    print(f"\nProcessing {emotion_name}")

    save_folder = output_root / emotion_name
    save_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    video_files = []

    video_files.extend(
        emotion_folder.glob("*.mov")
    )

    video_files.extend(
        emotion_folder.glob("*.MOV")
    )

    video_files.extend(
        emotion_folder.glob("*.mp4")
    )

    video_files.extend(
        emotion_folder.glob("*.MP4")
    )

    for video_file in video_files:

        print(f"  Video : {video_file.name}")

        cap = cv2.VideoCapture(
            str(video_file)
        )

        fps = int(
            cap.get(cv2.CAP_PROP_FPS)
        )

        if fps == 0:
            print("FPS gagal dibaca")
            continue

        frame_idx = 0
        saved_idx = 0

        while True:

            ret, frame = cap.read()

            if not ret:
                break

            # 1 FRAME SETIAP 1 DETIK

            if frame_idx % fps == 0:

                frame_name = (
                    f"{video_file.stem}_"
                    f"{saved_idx:04d}.jpg"
                )

                cv2.imwrite(
                    str(save_folder / frame_name),
                    frame
                )

                saved_idx += 1

            frame_idx += 1

        cap.release()

        print(
            f"    Saved : {saved_idx}"
        )

print("\nSELESAI")


Processing bingung
  Video : kiri.mov
    Saved : 65
  Video : tengah.mov
    Saved : 115
  Video : kanan.mov
    Saved : 65

Processing fokus
  Video : kiri.mov
    Saved : 54
  Video : tengah.mov
    Saved : 119
  Video : kanan.mov
    Saved : 51

Processing senang
  Video : kiri.mov
    Saved : 58
  Video : tengah.mov
    Saved : 121
  Video : kanan.mov
    Saved : 59

Processing bosan
  Video : kiri.mov
    Saved : 61
  Video : tengah.mov
    Saved : 127
  Video : kanan.mov
    Saved : 57

SELESAI


In [ ]:
!pip install ImageHash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.2 MB/s eta 0:00:00


In [ ]:
import shutil
from pathlib import Path

from PIL import Image
import imagehash

# ==========================
# PATH
# ==========================

input_root = Path("/content/drive/MyDrive/dataset-frame-raw")

output_root = Path("/content/drive/MyDrive/dataset-frame-selected")

output_root.mkdir(parents=True, exist_ok=True)

# ==========================
# PARAMETER
# ==========================

TARGET_PER_ANGLE = 30

HASH_THRESHOLD = 5

ANGLES = [
    "kiri",
    "tengah",
    "kanan"
]

# ==========================
# PROCESS
# ==========================

for emotion_folder in input_root.iterdir():

    if not emotion_folder.is_dir():
        continue

    emotion = emotion_folder.name

    print(f"\n========== {emotion.upper()} ==========")

    output_folder = output_root / emotion
    output_folder.mkdir(parents=True, exist_ok=True)

    total_saved = 0

    for angle in ANGLES:

        print(f"\nAngle: {angle}")

        # Ambil frame sesuai angle
        image_files = sorted(
            emotion_folder.glob(f"{angle}_*.jpg")
        )

        print(f"Total frame awal: {len(image_files)}")

        selected_images = []
        selected_hashes = []

        # ==========================
        # REMOVE DUPLICATE
        # ==========================

        for img_path in image_files:

            try:

                img = Image.open(img_path)

                current_hash = imagehash.phash(img)

                is_duplicate = False

                for prev_hash in selected_hashes:

                    distance = current_hash - prev_hash

                    if distance <= HASH_THRESHOLD:
                        is_duplicate = True
                        break

                if not is_duplicate:

                    selected_images.append(img_path)
                    selected_hashes.append(current_hash)

            except Exception:

                print(f"Error: {img_path.name}")

        print(
            f"Unique frame: {len(selected_images)}"
        )

        # ==========================
        # PILIH 30 FRAME MERATA
        # ==========================

        if len(selected_images) > TARGET_PER_ANGLE:

            step = len(selected_images) / TARGET_PER_ANGLE

            final_selection = []

            for i in range(TARGET_PER_ANGLE):

                idx = int(i * step)

                final_selection.append(
                    selected_images[idx]
                )

        else:

            final_selection = selected_images

        # ==========================
        # COPY KE FOLDER BARU
        # ==========================

        for img_path in final_selection:

            shutil.copy(
                img_path,
                output_folder / img_path.name
            )

        print(
            f"Selected: {len(final_selection)}"
        )

        total_saved += len(final_selection)

    print(
        f"\nTOTAL {emotion}: {total_saved} frame"
    )

print("\nSELESAI")


========== BINGUNG ==========

Angle: kiri
Total frame awal: 65
Unique frame: 11
Selected: 11

Angle: tengah
Total frame awal: 115
Unique frame: 17
Selected: 17

Angle: kanan
Total frame awal: 65
Unique frame: 11
Selected: 11

TOTAL bingung: 39 frame

========== FOKUS ==========

Angle: kiri
Total frame awal: 54
Unique frame: 18
Selected: 18

Angle: tengah
Total frame awal: 119
Unique frame: 16
Selected: 16

Angle: kanan
Total frame awal: 51
Unique frame: 17
Selected: 17

TOTAL fokus: 51 frame

========== SENANG ==========

Angle: kiri
Total frame awal: 58
Unique frame: 9
Selected: 9

Angle: tengah
Total frame awal: 121
Unique frame: 9
Selected: 9

Angle: kanan
Total frame awal: 59
Unique frame: 6
Selected: 6

TOTAL senang: 24 frame

========== BOSAN ==========

Angle: kiri
Total frame awal: 61
Unique frame: 22
Selected: 22

Angle: tengah
Total frame awal: 127
Unique frame: 14
Selected: 14

Angle: kanan
Total frame awal: 57
Unique frame: 19
Selected: 19

TOTAL bosan: 55 frame

SELESAI

In [ ]:
from pathlib import Path
import shutil

input_root = Path(
    "/content/drive/MyDrive/dataset-frame-raw"
)

output_root = Path(
    "/content/drive/MyDrive/dataset-frame-selected_ver2"
)

output_root.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_PER_ANGLE = 30

angles = [
    "kiri",
    "tengah",
    "kanan"
]

for emotion_folder in input_root.iterdir():

    if not emotion_folder.is_dir():
        continue

    emotion = emotion_folder.name

    print(f"\n========== {emotion.upper()} ==========")

    save_folder = output_root / emotion

    save_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    total_saved = 0

    for angle in angles:

        angle_images = sorted(
            emotion_folder.glob(
                f"{angle}_*.jpg"
            )
        )

        print(
            f"{angle}: {len(angle_images)} frame"
        )

        if len(angle_images) == 0:
            continue

        # pilih 30 frame merata
        step = len(angle_images) / TARGET_PER_ANGLE

        selected = []

        for i in range(TARGET_PER_ANGLE):

            idx = int(i * step)

            if idx >= len(angle_images):
                idx = len(angle_images) - 1

            selected.append(
                angle_images[idx]
            )

        for img_path in selected:

            shutil.copy(
                img_path,
                save_folder / img_path.name
            )

        total_saved += len(selected)

        print(
            f"Selected: {len(selected)}"
        )

    print(
        f"TOTAL {emotion}: {total_saved}"
    )

print("\nSELESAI")


========== BINGUNG ==========
kiri: 65 frame
Selected: 30
tengah: 115 frame
Selected: 30
kanan: 65 frame
Selected: 30
TOTAL bingung: 90

========== FOKUS ==========
kiri: 54 frame
Selected: 30
tengah: 119 frame
Selected: 30
kanan: 51 frame
Selected: 30
TOTAL fokus: 90

========== SENANG ==========
kiri: 58 frame
Selected: 30
tengah: 121 frame
Selected: 30
kanan: 59 frame
Selected: 30
TOTAL senang: 90

========== BOSAN ==========
kiri: 61 frame
Selected: 30
tengah: 127 frame
Selected: 30
kanan: 57 frame
Selected: 30
TOTAL bosan: 90

SELESAI


In [ ]:
from pathlib import Path
import shutil

# Folder asli
source_folder = Path(
    "/content/drive/MyDrive/dataset-frame-selected_ver2"
)

# Folder hasil copy
destination_folder = Path(
    "/content/drive/MyDrive/dataset-frame-selected_ver2_copy"
)

# Jika folder tujuan sudah ada, hapus dulu
if destination_folder.exists():
    shutil.rmtree(destination_folder)

# Copy seluruh folder
shutil.copytree(
    source_folder,
    destination_folder
)

print("Duplikasi selesai!")
print(f"Asal : {source_folder}")
print(f"Copy : {destination_folder}")

Duplikasi selesai!
Asal : /content/drive/MyDrive/dataset-frame-selected_ver2
Copy : /content/drive/MyDrive/dataset-frame-selected_ver2_copy


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/dataset-frame-selected_ver2")

for emotion in root.iterdir():
    if emotion.is_dir():
        print(emotion.name)
        print(list(emotion.glob("*.jpg"))[:5])
        print()

bingung
[PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/bingung/kiri_0000.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/bingung/kiri_0002.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/bingung/kiri_0004.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/bingung/kiri_0006.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/bingung/kiri_0008.jpg')]

fokus
[PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/fokus/kiri_0000.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/fokus/kiri_0001.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/fokus/kiri_0003.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/fokus/kiri_0005.jpg'), PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/fokus/kiri_0007.jpg')]

senang
[PosixPath('/content/drive/MyDrive/dataset-frame-selected_ver2/senang/kiri_0000.jpg'), PosixPath('/content/drive/My

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/dataset-frame-selected_ver2")

for emotion in root.iterdir():

    if emotion.is_dir():

        total = len(list(emotion.glob("*.jpg")))

        print(
            f"{emotion.name}: {total} frame"
        )

bingung: 90 frame
fokus: 90 frame
senang: 90 frame
bosan: 90 frame


In [ ]:
# Mengimpor Path untuk mengelola alamat folder dan file
from pathlib import Path

# Mengimpor shutil untuk menyalin file gambar
import shutil


# Menentukan folder utama yang berisi seluruh frame mentah
input_root = Path(
    "/content/drive/MyDrive/dataset-frame-raw"
)


# Menentukan folder tujuan untuk menyimpan frame yang sudah dipilih
output_root = Path(
    "/content/drive/MyDrive/datasetframe-selected150"
)


# Membuat folder output jika folder tersebut belum tersedia
output_root.mkdir(
    parents=True,   # Membuat seluruh folder induk jika belum tersedia
    exist_ok=True   # Tidak menghasilkan error jika folder sudah tersedia
)


# Menentukan jumlah frame yang akan diambil dari setiap sudut kamera
TARGET_PER_ANGLE = 50


# Daftar sudut kamera yang akan diproses
angles = [
    "kiri",    # Sudut kamera kiri
    "tengah",  # Sudut kamera tengah
    "kanan"    # Sudut kamera kanan
]


# Melakukan perulangan pada setiap folder emosi di dalam folder input
for emotion_folder in input_root.iterdir():

    # Memeriksa apakah item yang ditemukan merupakan folder
    if not emotion_folder.is_dir():

        # Melewati item tersebut jika bukan folder
        continue

    # Mengambil nama folder sebagai nama kelas emosi
    # Contohnya: fokus, senang, bosan, atau bingung
    emotion = emotion_folder.name

    # Menampilkan nama emosi yang sedang diproses
    print(f"\n========== {emotion.upper()} ==========")

    # Membuat alamat folder tujuan berdasarkan nama emosi
    save_folder = output_root / emotion

    # Membuat folder tujuan untuk kelas emosi tersebut
    save_folder.mkdir(
        parents=True,   # Membuat folder induk jika belum tersedia
        exist_ok=True   # Tidak error jika folder sudah tersedia
    )

    # Menyimpan jumlah keseluruhan frame yang berhasil dipilih
    # untuk satu kelas emosi
    total_saved = 0

    # Melakukan perulangan pada setiap sudut kamera
    for angle in angles:

        # Mengambil seluruh gambar JPG berdasarkan nama sudut kamera
        # Contohnya: kiri_1.jpg, kiri_2.jpg, dan seterusnya
        angle_images = sorted(
            emotion_folder.glob(
                f"{angle}_*.jpg"
            )
        )

        # Menampilkan nama sudut kamera yang sedang diproses
        print(
            f"\nAngle: {angle}"
        )

        # Menampilkan jumlah frame awal yang ditemukan pada sudut tersebut
        print(
            f"Total frame awal: {len(angle_images)}"
        )

        # Memeriksa apakah tidak ada gambar pada sudut kamera tersebut
        if len(angle_images) == 0:

            # Melewati sudut tersebut jika tidak ditemukan gambar
            continue

        # Jika jumlah frame lebih sedikit atau sama dengan target,
        # seluruh frame akan dipilih
        if len(angle_images) <= TARGET_PER_ANGLE:

            # Memilih seluruh gambar yang tersedia
            selected = angle_images

        # Jika jumlah frame lebih banyak daripada target
        else:

            # Menghitung jarak pengambilan frame agar tersebar merata
            # Contoh: 500 frame / 50 target = mengambil setiap 10 frame
            step = len(angle_images) / TARGET_PER_ANGLE

            # Membuat list kosong untuk menyimpan gambar terpilih
            selected = []

            # Melakukan perulangan sebanyak jumlah target frame
            for i in range(TARGET_PER_ANGLE):

                # Menghitung indeks gambar yang akan dipilih
                # int digunakan untuk mengubah hasil menjadi bilangan bulat
                idx = int(i * step)

                # Memasukkan gambar berdasarkan indeks ke dalam list selected
                selected.append(
                    angle_images[idx]
                )

        # Melakukan perulangan pada seluruh gambar yang telah dipilih
        for img_path in selected:

            # Menyalin gambar dari folder awal ke folder tujuan
            shutil.copy(
                img_path,                    # Lokasi gambar sumber
                save_folder / img_path.name  # Lokasi dan nama file tujuan
            )

        # Menambahkan jumlah gambar terpilih ke total frame emosi
        total_saved += len(selected)

        # Menampilkan jumlah gambar yang dipilih dari sudut tersebut
        print(
            f"Selected: {len(selected)}"
        )

    # Menampilkan jumlah keseluruhan frame untuk satu kelas emosi
    # Target maksimalnya adalah 150 frame:
    # 50 kiri + 50 tengah + 50 kanan
    print(
        f"\nTOTAL {emotion}: {total_saved} frame"
    )


# Menampilkan informasi bahwa seluruh proses telah selesai
print("\nSELESAI")


========== BINGUNG ==========

Angle: kiri
Total frame awal: 65
Selected: 50

Angle: tengah
Total frame awal: 115
Selected: 50

Angle: kanan
Total frame awal: 65
Selected: 50

TOTAL bingung: 150 frame

========== FOKUS ==========

Angle: kiri
Total frame awal: 54
Selected: 50

Angle: tengah
Total frame awal: 119
Selected: 50

Angle: kanan
Total frame awal: 51
Selected: 50

TOTAL fokus: 150 frame

========== SENANG ==========

Angle: kiri
Total frame awal: 58
Selected: 50

Angle: tengah
Total frame awal: 121
Selected: 50

Angle: kanan
Total frame awal: 59
Selected: 50

TOTAL senang: 150 frame

========== BOSAN ==========

Angle: kiri
Total frame awal: 61
Selected: 50

Angle: tengah
Total frame awal: 127
Selected: 50

Angle: kanan
Total frame awal: 57
Selected: 50

TOTAL bosan: 150 frame

SELESAI
